213. House Robber II

You are a professional robber planning to rob houses along a street. Each house has a certain amount of money stashed. All houses at this place are arranged in a circle. That means the first house is the neighbor of the last one. Meanwhile, adjacent houses have a security system connected, and it will automatically contact the police if two adjacent houses were broken into on the same night.

Given an integer array nums representing the amount of money of each house, return the maximum amount of money you can rob tonight without alerting the police.

 

Example 1:

Input: nums = [2,3,2]
Output: 3
Explanation: You cannot rob house 1 (money = 2) and then rob house 3 (money = 2), because they are adjacent houses.
Example 2:

Input: nums = [1,2,3,1]
Output: 4
Explanation: Rob house 1 (money = 1) and then rob house 3 (money = 3).
Total amount you can rob = 1 + 3 = 4.
Example 3:

Input: nums = [1,2,3]
Output: 3

## 1️⃣ Specification (Especificação de Engenharia)

**Core do Problema (2 frases):**
Dado um array circular de casas com valores, maximize o roubo sem Adjacência usando DP.
A restrição circular (primeira e última são vizinhas) exige decomposição em dois subcasos lineares.

**Blueprint (Python Type Hints):**
```python
from typing import List

class Solution:
    def rob(self, nums: List[int]) -> int:
        def rob_linear(houses: List[int]) -> int:
            # Resolve House Robber I (linear)
            ...
        # Trata a circularidade escolhendo qual extremo descartar
        ...
```

**Edge Cases & Pegadinhas:**
1. **Array vazio** (`[]`): Deve retornar 0 (sem casas = sem dinheiro).
2. **Uma casa só** (`[10]`): Roube ela, resultado = 10.
3. **Duas casas** (`[2, 5]`): Roube a maior → 5 (não pode roubar as duas, são vizinhas).
4. **Todos valores iguais** (`[5, 5, 5, 5]`): Máximo descontando uma extremidade.
5. **Valores extremos** (`[10^9, 1, 10^9]`): Cuidado com overflow (não há em Python), mas escolher bem qual house descartar é crítico.

---

## 2️⃣ Plan (A Heurística e o Desenho Técnico)

**Brute Force (Intuição — O que não fazer):**
Tentar todas as $2^N$ combinações de casas não-adjacentes.
```
Para [1, 2, 3, 1]:
Combinar [1], [1,3], [1,3,1] ← INVÁLIDO (1 e 1 são adjacentes em círculo)
Combinar [2], [2,3], ... 
Explosão combinatorial → O(2^N)
```

**Gargalo:**
Força bruta repete subproblemas (ex: "qual é o máximo até a casa 2?") milhares de vezes.
Sem memoização, é inviável para N > 20.

**Heurística Otimizada (DP + Redução Circular):**
- **Insight:**  Em um círculo, a primeira e última casa são vizinhas.
- **Estratégia:**  Quebre a circularidade em **dois casos lineares**:
  1. **Caso 1:** Roube a primeira casa → não pode roubar a última → resolve `nums[:-1]`
  2. **Caso 2:** Roube a última casa → não pode roubar a primeira → resolve `nums[1:]`
  3. **Resultado:** `max(caso1, caso2)`
- **Por que vence:**  Reduz de O(2^N) → O(N) usando DP linear 2x.

**Visual Trace (Exemplo Didático):**

Dado: `nums = [2, 3, 2]`

```
Estrutura circular:
    2 ←  3  
    ↓    ↑
    2 ────

Caso 1: Excluir última (rouba primeira se quiser)
        [2, 3]  → máximo = 3 (rouba 3)
        
Caso 2: Excluir primeira (rouba última se quiser)
        [3, 2]  → máximo = 3 (rouba 3)
        
Resposta: max(3, 3) = 3 ✓
```

**DP Table (Exemplo com [1, 2, 3, 1]):**

Caso 1: `nums[:-1]` = `[1, 2, 3]`
```
i:     0    1    2  
nums: [1    2    3]
dp:   [1    2    4]
      └─ rouba só 1
         └─ rouba só 2 (melhor que 1)
            └─ rouba 1+3 (melhor que 2)
```
Máximo Caso 1: **4**

Caso 2: `nums[1:]` = `[2, 3, 1]`
```
i:     0    1    2  
nums: [2    3    1]
dp:   [2    3    3]
      └─ rouba só 2
         └─ rouba só 3 (melhor que 2)
            └─ rouba 2+1 = 3 (igual a 3)
```
Máximo Caso 2: **3**

**Resultado Final:** `max(4, 3) = 4` ✓

In [ ]:
from typing import List

class Solution:
    def rob(self, nums: List[int]) -> int:
        """
        House Robber II com DP em O(1) space.
        Estratégia: Quebra círculo em dois casos lineares.
        """
        # Edge case 1: Nenhuma casa
        if not nums:
            return 0  # O(1)
        
        # Edge case 2: Uma casa só
        if len(nums) == 1:
            return nums[0]  # O(1)
        
        def rob_linear(houses: List[int]) -> int:
            """
            Resolve House Robber I (versão linear).
            Usa apenas 2 variáveis para rastrear máximo até i-1 e i-2.
            
            Lógica: Para cada casa, escolha:
            - Roubá-la: prev + house_value
            - Não roubar: curr (máximo até casa anterior)
            """
            prev, curr = 0, 0  # prev=dp[i-2], curr=dp[i-1]  # O(1)
            
            # Itera sobre cada casa no intervalo
            for house_value in houses:
                # Atualização simultânea de ambas as variáveis
                # Novo curr = max(não roubou, roubou)
                prev, curr = curr, max(curr, prev + house_value)  # O(1) por iteração
            
            return curr  # O(1)
        
        # DECISÃO CRÍTICA: Por que dois casos?
        # Em círculo, primeira e última são vizinhas.
        # Não podemos roubar ambas → dois cenários mutuamente exclusivos.
        
        # Caso 1: Exclui a última casa (permite roubá-la ou não)
        case_exclude_last = rob_linear(nums[:-1])  # O(N)
        
        # Caso 2: Exclui a primeira casa (permite roubar última ou não)
        case_exclude_first = rob_linear(nums[1:])  # O(N)
        
        # Resposta: melhor entre os dois cenários
        return max(case_exclude_last, case_exclude_first)  # O(1)

In [ ]:

# 🧪 Testes da Solução House Robber II

sol = Solution()

tests = [
    ([2, 3, 2], 3, "Exemplo 1: Não pode roubar 2 e 2 (vizinhas)"),
    ([1, 2, 3, 1], 4, "Exemplo 2: Rouba 1 e 3"),
    ([1, 2, 3], 3, "Exemplo 3: Rouba a última (3)"),
    ([10], 10, "Edge case: Uma casa só"),
    ([], 0, "Edge case: Sem casas"),
    ([2, 5], 5, "Edge case: Duas casas (rouba a maior)"),
    ([5, 5, 5, 5], 10, "Todos iguais: Alterna entre casas"),
]

print("=" * 80)
print("🎯 HOUSE ROBBER II - TEST SUITE")
print("=" * 80)

passed = 0
for nums, expected, description in tests:
    result = sol.rob(nums)
    status = "✓ PASS" if result == expected else "✗ FAIL"
    if result == expected:
        passed += 1
    print(f"\n{status} | {description}")
    print(f"   Input:    {nums}")
    print(f"   Expected: {expected}, Got: {result}")

print("\n" + "=" * 80)
print(f"📊 RESULTADO: {passed}/{len(tests)} testes passaram")
print("=" * 80)

---

## 3️⃣ Implementation (Comentários de Decisão)

**Por que `prev, curr = curr, max(curr, prev + house_value)`?**
- Atualização simultânea: evita usar variáveis temporárias
- Pythônico: elegância e legibilidade
- Equivalente a `temp = curr; curr = max(...); prev = temp` (3 linhas vs 1)

**Por que NÃO usar `dp = [0] * len(houses)`?**
- Desperdiça O(N) espaço quando só precisamos dos 2 últimos valores
- Menos didático em performance crítica (ex: sistemas distribuídos)

**Por que dois `rob_linear()` calls separadas?**
- Clareza: cada caso é uma variante do problema linear
- Reutilização: o mesmo algoritmo resolve ambos
- Não perde eficiência: O(N) + O(N) = O(N)

---

## 4️⃣ Complexity (Veredito Final)

**Time Complexity: O(N)**
- `rob_linear(nums[:-1])` → itera N-1 casas → O(N-1) = O(N)
- `rob_linear(nums[1:])` → itera N-1 casas → O(N-1) = O(N)
- `max(case1, case2)` → O(1)
- **Total:** O(N) + O(N) + O(1) = **O(N)** ✓

**Space Complexity: O(1)** 
- Variáveis: `prev, curr` → O(1)
- Slices `nums[:-1]` e `nums[1:]` → O(1) iteradores (não copiam na memória durante for loop)
- Recursão: nenhuma
- **Total:** **O(1)** auxiliar ✓ (excepto espaço da entrada)

**Resumo:**
| Métrica | Valor | Razão |
|---------|-------|-------|
| Time | O(N) | 2 passagens lineares |
| Space | O(1) | Apenas 2 variáveis |
| Otimização | DP + Decomposição | Quebra círculo em 2 casos lineares |

---

**Dica Final (Live Coding Interview):**
Se a entrevistadora disser "sem slices", use `rob_linear(0, n-1)` e `rob_linear(1, n)` com parâmetros de intervalo. Mesma complexidade, sem criar slices intermediários! 🎯